In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.5219999999999999, 10: 0.525, 20: 0.5315000000000001, 30: 0.5410000000000001, 40: 0.5469999999999997, 50: 0.5454999999999999, 60: 0.5469999999999998, 70: 0.5530000000000002, 80: 0.5524999999999998, 90: 0.5569999999999999, 100: 0.5664999999999998, 110: 0.5729999999999997, 120: 0.583, 130: 0.575, 140: 0.579, 150: 0.5929999999999997, 160: 0.5915, 170: 0.607, 180: 0.6025, 190: 0.6089999999999999, 200: 0.6144999999999999, 210: 0.607, 220: 0.6144999999999999, 230: 0.6209999999999999, 240: 0.6275, 250: 0.63, 260: 0.6389743589743591, 270: 0.6491891891891892, 280: 0.6378378378378378, 290: 0.6470270270270271, 300: 0.6437837837837838}
{0: 0.005876, 10: 0.005015, 20: 0.0052777499999999995, 30: 0.005759000000000001, 40: 0.006111, 50: 0.00559975, 60: 0.006191000000000001, 70: 0.005870999999999999, 80: 0.00649375, 90: 0.006611, 100: 0.004967749999999999, 110: 0.005210999999999999, 120: 0.006351, 130: 0.006195, 140: 0.0057789999999999985, 150: 0.006951000000000001, 160: 0.006217749999999999, 170: